# WarehouseSort — End-to-End RGB-State Diffusion Policy Solution

This notebook trains and evaluates the end-to-end **RGB-State Diffusion Policy** on all three difficulty levels:
**Easy**, **Medium**, and **Hard**.

Under the updated rules, we include state information (object poses, tag colors, bin positions, and colors) in the observation space, which are automatically flattened into the `obs["state"]` vector. The policy maps this augmented state vector and camera images directly to actions using a Conditional 1D U-Net.

To meet strict execution time limitations, this notebook uses a highly optimized training pipeline:
1. **PlainConv Visual Encoder**: A lightweight convolutional feature extractor instead of ResNet-18.
2. **Reduced Trajectory Count**: Uses 50 (Easy) / 100 (Medium) / 150 (Hard) demonstration episodes.
3. **Shortened Iteration Count**: Runs 8k (Easy) / 12k (Medium) / 16k (Hard) steps.
4. **No Video Compression**: Disables evaluation video saving during training to eliminate ffmpeg/io bottleneck.

**Requirements:** a CUDA GPU. In Google Colab: *Runtime → Change runtime type → T4 GPU*.

## 1. Installation & Headless Render Setup

In [ ]:
# Install ManiSkill and dependencies (takes ~2 min)
!pip install mani-skill==3.0.1 diffusers==0.38.0 gymnasium torch torchvision hydra-core -q

# Install the warehouse_sort package
!pip install -e . -q

# Colab headless rendering (offscreen Vulkan)
import os
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'

## 2. Level 1: Easy (2 Parcels, Fixed Positions)
Trains in ~2-4 minutes on a T4 GPU.

In [ ]:
# 1. Generate 50 demonstration episodes
!python il/gen_demos.py --difficulty easy --num-episodes 50 --obs-modes rgb

# 2. Train the RGB-State Diffusion Policy
!python il/train.py method=dp_rgb demo_dir=easy \
    flags.total_iters=8000 \
    flags.eval_freq=4000 \
    flags.exp_name=warehouse_rgb_dp_easy

# 3. Evaluate the trained Easy policy
!python eval.py difficulty=easy obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp_easy/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

## 3. Level 2: Medium (4 Parcels, Jittered Positions)
Trains in ~4-6 minutes on a T4 GPU.

In [ ]:
# 1. Generate 100 demonstration episodes
!python il/gen_demos.py --difficulty medium --num-episodes 100 --obs-modes rgb

# 2. Train the RGB-State Diffusion Policy
!python il/train.py method=dp_rgb demo_dir=medium \
    flags.total_iters=12000 \
    flags.eval_freq=4000 \
    flags.exp_name=warehouse_rgb_dp_medium

# 3. Evaluate the trained Medium policy
!python eval.py difficulty=medium obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp_medium/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

## 4. Level 3: Hard (6 Parcels, Jittered Positions & Swapping Bins)
Trains in ~6-8 minutes on a T4 GPU.

In [ ]:
# 1. Generate 150 demonstration episodes
!python il/gen_demos.py --difficulty hard --num-episodes 150 --obs-modes rgb

# 2. Train the RGB-State Diffusion Policy
!python il/train.py method=dp_rgb demo_dir=hard \
    flags.total_iters=16000 \
    flags.eval_freq=4000 \
    flags.exp_name=warehouse_rgb_dp_hard

# 3. Evaluate the trained Hard policy
!python eval.py difficulty=hard obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp_hard/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml